# 🛰️ SatQuery AI — Multi-Session Continuous Multimodal Fusion Pipeline
**Problem Statement ID:** 26167 (ISRO / SAC)  
**Dataset:** `nightwingggg/yugkitatti` (`BigEarthNet-S1.tar.zst` + `BigEarthNet-S2.tar.zst` + `metadata.parquet`)  
**Input Datasets:**
- `yugkitatti`: Base dataset (`BigEarthNet-S1.tar.zst`, `BigEarthNet-S2.tar.zst`, `metadata.parquet`)
- `progress`: Previous session checkpoints (`fusion_resnet50_latest.pt`, `fusion_resnet50_best.pt`, `fusion_training_history.json`)
**Architecture:** Hybrid Multimodal Network combining **ResNet-50 MoCo SAR Radar** + **ConvNeXt-Tiny Optical (RGB+NIR)** via **Gated Cross-Sensor Attention**.

---
### ⚡ Multi-Session Continuous Shard Architecture:
1. **Dynamic Global Sharding:** Automatically partitions the entire attached dataset into discrete, deterministically shuffled global shards (~35,000 paired patches = ~6 GB each).
2. **Session Windowing (6 Shards Per Session):** Each Kaggle session trains a window of up to 6 shards (e.g. Session 1: Shards 1–6; Session 2: Shards 7–12; Session 3: Shards 13+).
3. **Seamless State Continuity & LR Reset Fix:** Preserves model weights and optimizer momentum, while properly resetting the learning rate per shard to prevent scheduler freeze (`base_lrs = 0.0`).
4. **Smart Checkpoint Auto-Detection:** Automatically scans `/kaggle/input` across attached datasets (`progress` / models), inspects stored metadata, and stages checkpoints and history seamlessly.
5. **Shard Override Control:** Allows pinpointing `FORCE_START_SHARD = 3` so you can jump straight into Shard 3 with trained weights without starting from scratch.
6. **Disk-Safe Shard Rotation:** Extracts 1 shard (~6 GB) into `/tmp/fusion_data` at a time, trains 12 epochs, then wipes `/tmp` before extracting the next shard. Disk usage never exceeds ~7 GB.


In [ ]:
# 1. Install necessary dependencies
!apt-get update -qq && apt-get install -y -qq zstd
!pip install -q timm zstandard pyarrow rasterio torchgeo scikit-learn


In [ ]:
# 2. Verify GPU & Hardware Acceleration
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    major, _ = torch.cuda.get_device_capability(0)
    print(f"GPU: {gpu_name} ({total_vram:.2f} GB VRAM, Compute Capability {major})")
    
    # T4-Safe settings: disable cuDNN benchmark to prevent depthwise engine search failures
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("⚡ Hardware acceleration initialized safely for Tesla T4.")
else:
    print("⚠️ NO GPU DETECTED. Please enable GPU in Kaggle Settings -> Accelerator.")


In [ ]:
# 3. Paths, Hyperparameters, Global Sharding & Smart Checkpoint Resume
import os
import sys
import time
import json
import glob
import shutil
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# --- Wall-Clock Runtime Safeguard ---
MAX_RUNTIME_HOURS = 8.5
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600
SESSION_START_TIME = time.time()

# --- Multi-Session Sharding Hyperparameters ---
SHARDS_PER_SESSION = 6          # Number of shards to train per Kaggle run
SAMPLES_PER_SHARD = 35000       # 35,000 paired patches per shard (~6 GB disk per shard)
EPOCHS_PER_SHARD = 12           # 12 epochs per shard
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Shard Resume Override ---
# Set to 3 to resume directly at Global Shard 3 with your existing weights!
# Set to None for automatic resume based on the checkpoint's stored epoch/shard.
FORCE_START_SHARD = None

# --- Locate BigEarthNet Datasets in /kaggle/input ---
print("🔎 Locating BigEarthNet-S1 and S2 archives in /kaggle/input...")
s1_archives = glob.glob("/kaggle/input/**/BigEarthNet-S1.tar.zst", recursive=True)
s2_archives = glob.glob("/kaggle/input/**/BigEarthNet-S2.tar.zst", recursive=True)

if not s1_archives or not s2_archives:
    raise FileNotFoundError("Both BigEarthNet-S1.tar.zst and BigEarthNet-S2.tar.zst must be attached!")

S1_ARCHIVE = s1_archives[0]
S2_ARCHIVE = s2_archives[0]
DATASET_ROOT = os.path.dirname(S1_ARCHIVE)
METADATA_PATH = os.path.join(DATASET_ROOT, "metadata.parquet")
METADATA_EXCLUDED_PATH = os.path.join(DATASET_ROOT, "metadata_for_patches_with_snow_cloud_or_shadow.parquet")

print(f"✅ S1 Archive: {S1_ARCHIVE}")
print(f"✅ S2 Archive: {S2_ARCHIVE}")

# Output directories
OUTPUT_DIR = "/kaggle/working"
TMP_FUSION_DIR = "/tmp/fusion_data"
S1_EXTRACT_DIR = os.path.join(TMP_FUSION_DIR, "s1")
S2_EXTRACT_DIR = os.path.join(TMP_FUSION_DIR, "s2")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(S1_EXTRACT_DIR, exist_ok=True)
os.makedirs(S2_EXTRACT_DIR, exist_ok=True)

LATEST_CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "fusion_resnet50_latest.pt")
BEST_CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "fusion_resnet50_best.pt")
ZIP_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "fusion_model_artifacts.zip")
HISTORY_PATH = os.path.join(OUTPUT_DIR, "fusion_training_history.json")
CLASSES_PATH = os.path.join(OUTPUT_DIR, "classes.json")

CORINE_19_CLASSES = [
    'Agro-forestry areas', 'Arable land', 'Beaches, dunes, sands', 'Broad-leaved forest',
    'Coastal wetlands', 'Complex cultivation patterns', 'Coniferous forest',
    'Industrial or commercial units', 'Inland waters', 'Inland wetlands',
    'Land principally occupied by agriculture, with significant areas of natural vegetation',
    'Marine waters', 'Mixed forest', 'Moors, heathland and sclerophyllous vegetation',
    'Natural grassland and sparsely vegetated areas', 'Pastures', 'Permanent crops',
    'Transitional woodland, shrub', 'Urban fabric'
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CORINE_19_CLASSES)}

with open(CLASSES_PATH, 'w') as f:
    json.dump({"classes": CORINE_19_CLASSES, "class_to_idx": CLASS_TO_IDX}, f, indent=2)

# --- Smart Auto-Detection of Checkpoints from Attached Datasets ('progress') ---
def find_and_stage_best_checkpoint():
    if os.path.exists(LATEST_CHECKPOINT_PATH):
        print("ℹ️ Checkpoint already staged in /kaggle/working.")
        return

    # 1. Search for uploaded zip archives first
    candidate_zips = glob.glob("/kaggle/input/**/fusion_model_artifacts*.zip", recursive=True)
    if candidate_zips:
        candidate_zips.sort(key=lambda p: os.path.getmtime(p), reverse=True)
        chosen_zip = candidate_zips[0]
        print(f"📦 Found uploaded artifact zip: {chosen_zip}")
        print(f"   Unpacking into {OUTPUT_DIR}...")
        try:
            with zipfile.ZipFile(chosen_zip, 'r') as zf:
                zf.extractall(OUTPUT_DIR)
            print("   ✅ Successfully unpacked checkpoint artifacts!")
            return
        except Exception as e:
            print(f"   ⚠️ Could not unpack {chosen_zip}: {e}")

    # 2. Search for standalone .pt checkpoints across /kaggle/input (e.g. in /kaggle/input/progress/)
    candidate_pts = (
        glob.glob("/kaggle/input/progress/fusion_resnet50*.pt") +
        glob.glob("/kaggle/input/**/fusion_resnet50*.pt", recursive=True) +
        glob.glob("/kaggle/input/models/**/fusion_resnet50*.pt", recursive=True) +
        glob.glob("/kaggle/input/**/*.pt", recursive=True)
    )
    candidate_pts = list(dict.fromkeys([p for p in candidate_pts if os.path.isfile(p)]))
    
    if not candidate_pts:
        print("ℹ️ No previous checkpoints found in /kaggle/input. Starting brand-new training run.")
        return

    print(f"🔎 Evaluating {len(candidate_pts)} candidate .pt files in /kaggle/input...")
    best_candidate = None
    best_tuple = (-1, -1, False, 0.0)

    for pt_path in candidate_pts:
        try:
            mtime = os.path.getmtime(pt_path)
            fname = os.path.basename(pt_path).lower()
            is_latest_name = ('latest' in fname)
            
            ckpt_data = torch.load(pt_path, map_location='cpu', weights_only=False)
            if isinstance(ckpt_data, dict) and 'model_state_dict' in ckpt_data:
                s_idx = ckpt_data.get('shard_index', 0)
                ep = ckpt_data.get('epoch', 0)
                score = (s_idx, ep, is_latest_name, mtime)
                print(f"   • {pt_path} -> Shard {s_idx + 1}, Epoch {ep} ({'latest' if is_latest_name else 'other'})")
                if score > best_tuple:
                    best_tuple = score
                    best_candidate = pt_path
        except Exception as e:
            continue

    if best_candidate:
        print(f"🏆 Selected checkpoint: {best_candidate} (Global Shard {best_tuple[0] + 1}, Epoch {best_tuple[1]})")
        shutil.copy(best_candidate, LATEST_CHECKPOINT_PATH)
        if not os.path.exists(BEST_CHECKPOINT_PATH):
            shutil.copy(best_candidate, BEST_CHECKPOINT_PATH)
        print(f"✅ Staged as {LATEST_CHECKPOINT_PATH}")

        # Also stage previous training history from /kaggle/input/progress/ if available
        candidate_histories = glob.glob("/kaggle/input/**/fusion_training_history.json", recursive=True)
        if candidate_histories and not os.path.exists(HISTORY_PATH):
            try:
                shutil.copy(candidate_histories[0], HISTORY_PATH)
                print(f"✅ Staged previous training history from {candidate_histories[0]}")
            except Exception as e:
                print(f"⚠️ Could not stage training history: {e}")
    else:
        print("ℹ️ No valid multimodal fusion checkpoints found among candidates. Starting fresh.")

find_and_stage_best_checkpoint()

# --- Read and Prepare Matched Metadata Across Entire Dataset ---
print("📖 Ingesting dataset metadata to construct continuous global sharding index...")
meta_df = pd.read_parquet(METADATA_PATH)

if os.path.exists(METADATA_EXCLUDED_PATH):
    excluded_df = pd.read_parquet(METADATA_EXCLUDED_PATH)
    cloud_recoverable = excluded_df[(~excluded_df['contains_seasonal_snow']) & (excluded_df['contains_cloud_or_shadow'])].copy()
    print(f"⛅ Ingesting {len(cloud_recoverable):,} patches with optical cloud/shadow: teaches fusion network to route to SAR!")
    meta_df = pd.concat([meta_df, cloud_recoverable], ignore_index=True)

if 'contains_seasonal_snow' in meta_df.columns:
    meta_df = meta_df[~meta_df['contains_seasonal_snow']]

# Clean matched pairs
paired_df = meta_df[meta_df['patch_id'].notna() & meta_df['s1_name'].notna()].copy()
print(f"📊 Total clean paired candidates in dataset: {len(paired_df):,}")

# Deterministic global shuffle (seed 42) ensures identical shard boundaries across all restarts
shuffled_all = paired_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
TOTAL_GLOBAL_SHARDS = int(np.ceil(len(shuffled_all) / SAMPLES_PER_SHARD))

print(f"🎯 Partitioned entire dataset into {TOTAL_GLOBAL_SHARDS} Global Shards of up to {SAMPLES_PER_SHARD:,} pairs each.")
print(f"   Each session trains a window of up to {SHARDS_PER_SESSION} shards ({SHARDS_PER_SESSION * SAMPLES_PER_SHARD:,} scenes).")

shards_meta = []
for s_idx in range(TOTAL_GLOBAL_SHARDS):
    start_i = s_idx * SAMPLES_PER_SHARD
    end_i = min((s_idx + 1) * SAMPLES_PER_SHARD, len(shuffled_all))
    s_df = shuffled_all.iloc[start_i:end_i].copy().reset_index(drop=True)
    shards_meta.append(s_df)
    if s_idx < 6 or s_idx == TOTAL_GLOBAL_SHARDS - 1:
        print(f"   📁 Global Shard {s_idx + 1}/{TOTAL_GLOBAL_SHARDS}: {len(s_df):,} pairs (indices {start_i:,} to {end_i:,})")
    elif s_idx == 6:
        print(f"   ... [Global Shards 7 to {TOTAL_GLOBAL_SHARDS - 1} indexed identically] ...")


In [ ]:
# 4. Disk-Safe Shard Unpacking & Rotation Engine
import tarfile
import zstandard as zstd

def clear_fusion_tmp_dir():
    if os.path.exists(TMP_FUSION_DIR):
        shutil.rmtree(TMP_FUSION_DIR, ignore_errors=True)
    os.makedirs(S1_EXTRACT_DIR, exist_ok=True)
    os.makedirs(S2_EXTRACT_DIR, exist_ok=True)

def extract_shard_data(shard_df, shard_num, total_shards=TOTAL_GLOBAL_SHARDS):
    if len(shard_df) == 0:
        print(f"⚠️ Shard {shard_num} contains 0 samples — skipping extraction.")
        return

    done_flag = os.path.join(TMP_FUSION_DIR, f".shard_{shard_num}_done")
    if os.path.exists(done_flag):
        print(f"⚡ Shard {shard_num} data already extracted in /tmp/fusion_data.")
        return
        
    clear_fusion_tmp_dir()
    target_s1_names = set(shard_df['s1_name'].astype(str))
    target_s2_names = set(shard_df['patch_id'].astype(str))
    
    # 1. S1 Extraction with Early Exit & Progress
    print(f"📦 [Global Shard {shard_num}/{total_shards}] Extracting {len(target_s1_names):,} S1 radar patches into {S1_EXTRACT_DIR}...")
    extracted_s1 = 0
    t0 = time.time()
    dctx = zstd.ZstdDecompressor()
    
    with open(S1_ARCHIVE, 'rb') as fh:
        with dctx.stream_reader(fh) as stream:
            with tarfile.open(mode='r|', fileobj=stream) as tar:
                for member in tar:
                    if not member.isfile() or not member.name.endswith('.tif'):
                        continue
                    fname = os.path.basename(member.name)
                    s1_name = fname.replace("_VV.tif", "").replace("_VH.tif", "")
                    if s1_name in target_s1_names:
                        dest_path = os.path.join(S1_EXTRACT_DIR, fname)
                        if not os.path.exists(dest_path):
                            f = tar.extractfile(member)
                            if f:
                                with open(dest_path, 'wb') as out_f:
                                    out_f.write(f.read())
                                extracted_s1 += 1
                                if extracted_s1 % 10000 == 0:
                                    print(f"   ⏳ Extracted {extracted_s1:,} / {len(target_s1_names)*2:,} S1 bands ({time.time()-t0:.1f}s)...", flush=True)
                    # Early exit once both VV and VH bands are extracted
                    if extracted_s1 >= len(target_s1_names) * 2:
                        print(f"   ⚡ All {extracted_s1:,} S1 bands found! Early exit triggered.", flush=True)
                        break
                        
    print(f"   ✅ Shard {shard_num} S1 radar extraction complete in {time.time()-t0:.1f}s!")
    
    # 2. S2 Extraction with Early Exit & Progress
    print(f"📦 [Global Shard {shard_num}/{total_shards}] Extracting {len(target_s2_names):,} S2 optical patches (RGB+NIR) into {S2_EXTRACT_DIR}...")
    extracted_s2 = 0
    t0 = time.time()
    s2_bands_keep = ('_B02.tif', '_B03.tif', '_B04.tif', '_B08.tif')
    
    with open(S2_ARCHIVE, 'rb') as fh:
        with dctx.stream_reader(fh) as stream:
            with tarfile.open(mode='r|', fileobj=stream) as tar:
                for member in tar:
                    if not member.isfile() or not any(member.name.endswith(b) for b in s2_bands_keep):
                        continue
                    fname = os.path.basename(member.name)
                    patch_id = fname.split('_B0')[0] if '_B0' in fname else fname.split('_B')[0]
                    if patch_id in target_s2_names:
                        dest_path = os.path.join(S2_EXTRACT_DIR, fname)
                        if not os.path.exists(dest_path):
                            f = tar.extractfile(member)
                            if f:
                                with open(dest_path, 'wb') as out_f:
                                    out_f.write(f.read())
                                extracted_s2 += 1
                                if extracted_s2 % 15000 == 0:
                                    print(f"   ⏳ Extracted {extracted_s2:,} / {len(target_s2_names)*4:,} S2 bands ({time.time()-t0:.1f}s)...", flush=True)
                    # Early exit once all 4 bands are extracted
                    if extracted_s2 >= len(target_s2_names) * 4:
                        print(f"   ⚡ All {extracted_s2:,} S2 bands found! Early exit triggered.", flush=True)
                        break
                        
    print(f"   ✅ Shard {shard_num} S2 optical extraction complete in {time.time()-t0:.1f}s!")
    with open(done_flag, 'w') as f:
        f.write('done')
    
    total, used, free = shutil.disk_usage("/tmp")
    print(f"🎉 Shard {shard_num} /tmp disk: {used/(1024**3):.1f} GB used, {free/(1024**3):.1f} GB FREE.")


In [ ]:
# 5. Robust Multi-Modality Fusion Dataset & Label Parser
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import random
import ast

def extract_active_indices(raw_label, class_to_idx, num_classes=19):
    if raw_label is None:
        return []
    if isinstance(raw_label, str):
        raw_label = raw_label.strip()
        if raw_label.startswith('[') and raw_label.endswith(']'):
            try:
                raw_label = ast.literal_eval(raw_label)
            except Exception:
                raw_label = [x.strip(" '") for x in raw_label[1:-1].split(',')]
        else:
            raw_label = [x.strip(" '") for x in raw_label.split(',')]

    active_indices = []
    if hasattr(raw_label, '__iter__'):
        for item in raw_label:
            if isinstance(item, (int, np.integer)):
                if 0 <= int(item) < num_classes:
                    active_indices.append(int(item))
            elif isinstance(item, (str, np.str_)):
                s = str(item).strip()
                if s in class_to_idx:
                    active_indices.append(class_to_idx[s])
                else:
                    norm_s = s.lower().replace('-', ' ').replace('_', ' ')
                    for c_name, c_idx in class_to_idx.items():
                        if c_name.lower().replace('-', ' ').replace('_', ' ') == norm_s:
                            active_indices.append(c_idx)
                            break
    return sorted(list(set(active_indices)))

class BigEarthNetFusionDataset(Dataset):
    def __init__(self, df, s1_dir, s2_dir, is_train=True):
        self.s1_names = list(df['s1_name'].astype(str))
        self.s2_names = list(df['patch_id'].astype(str))
        self.labels = list(df['labels'])
        self.s1_dir = s1_dir
        self.s2_dir = s2_dir
        self.is_train = is_train
        self.length = len(self.s1_names)

    def __len__(self):
        return self.length

    def _load_s1(self, s1_name):
        vv_p = os.path.join(self.s1_dir, f"{s1_name}_VV.tif")
        vh_p = os.path.join(self.s1_dir, f"{s1_name}_VH.tif")
        vv = np.array(Image.open(vv_p), dtype=np.float32) if os.path.exists(vv_p) else np.full((120, 120), -25.0, dtype=np.float32)
        vh = np.array(Image.open(vh_p), dtype=np.float32) if os.path.exists(vh_p) else np.full((120, 120), -32.5, dtype=np.float32)
        
        vv = np.nan_to_num(vv, nan=-25.0, posinf=0.0, neginf=-45.0)
        vh = np.nan_to_num(vh, nan=-32.5, posinf=0.0, neginf=-45.0)
        ch0 = np.clip((vv + 25.0) / 25.0, 0.0, 1.0)
        ch1 = np.clip((vh + 32.5) / 32.5, 0.0, 1.0)
        return torch.from_numpy(np.stack([ch0, ch1], axis=0))

    def _load_s2(self, s2_name):
        bands = []
        for b_name in ['_B04.tif', '_B03.tif', '_B02.tif', '_B08.tif']:
            bp = os.path.join(self.s2_dir, f"{s2_name}{b_name}")
            if os.path.exists(bp):
                b_arr = np.array(Image.open(bp), dtype=np.float32)
                b_norm = np.clip(b_arr / 10000.0, 0.0, 1.0)
            else:
                b_norm = np.zeros((120, 120), dtype=np.float32)
            bands.append(b_norm)
        return torch.from_numpy(np.stack(bands, axis=0))

    def __getitem__(self, idx):
        s1_tensor = self._load_s1(self.s1_names[idx])
        s2_tensor = self._load_s2(self.s2_names[idx])

        if self.is_train:
            if random.random() > 0.5:
                s1_tensor = TF.hflip(s1_tensor)
                s2_tensor = TF.hflip(s2_tensor)
            if random.random() > 0.5:
                s1_tensor = TF.vflip(s1_tensor)
                s2_tensor = TF.vflip(s2_tensor)

        target = torch.zeros(len(CORINE_19_CLASSES), dtype=torch.float32)
        active_indices = extract_active_indices(self.labels[idx], CLASS_TO_IDX, len(CORINE_19_CLASSES))
        for c_idx in active_indices:
            target[c_idx] = 1.0

        return s1_tensor, s2_tensor, target


In [ ]:
# 6. Hybrid Cross-Sensor Gated Fusion Architecture
import timm
from torchgeo.models import ResNet50_Weights

class CrossSensorGatedFusion(nn.Module):
    def __init__(self, sar_dim=2048, opt_dim=768, fusion_dim=1024):
        super().__init__()
        self.proj_sar = nn.Sequential(nn.Linear(sar_dim, fusion_dim), nn.LayerNorm(fusion_dim))
        self.proj_opt = nn.Sequential(nn.Linear(opt_dim, fusion_dim), nn.LayerNorm(fusion_dim))
        self.gate = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim // 2),
            nn.GELU(),
            nn.Linear(fusion_dim // 2, fusion_dim),
            nn.Sigmoid()
        )
        self.fusion_norm = nn.LayerNorm(fusion_dim)

    def forward(self, f_sar, f_opt):
        h_sar = self.proj_sar(f_sar)
        h_opt = self.proj_opt(f_opt)
        concat_feat = torch.cat([h_sar, h_opt], dim=-1)
        alpha = self.gate(concat_feat)
        f_fused = alpha * h_sar + (1.0 - alpha) * h_opt
        return self.fusion_norm(f_fused), alpha

class SatQueryMultimodalFusionModel(nn.Module):
    def __init__(self, num_classes=19):
        super().__init__()
        self.sar_backbone = timm.create_model("resnet50", in_chans=2, num_classes=0)
        try:
            s1_weights = ResNet50_Weights.SENTINEL1_ALL_MOCO
            self.sar_backbone.load_state_dict(s1_weights.get_state_dict(progress=True), strict=False)
            print("   ✅ Branch 1: Pretrained MoCo SAR ResNet-50 loaded.")
        except Exception as e:
            print(f"   ℹ️ MoCo weights init fallback: {e}")
        
        self.opt_backbone = timm.create_model("convnext_tiny", in_chans=4, num_classes=0, pretrained=True)
        print("   ✅ Branch 2: Pretrained ConvNeXt-Tiny Optical loaded (4 channels RGB+NIR).")
        
        sar_dim = self.sar_backbone.num_features
        opt_dim = self.opt_backbone.num_features
        fusion_dim = 1024
        
        self.fusion_block = CrossSensorGatedFusion(sar_dim=sar_dim, opt_dim=opt_dim, fusion_dim=fusion_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(fusion_dim, num_classes)
        )

    def forward(self, x_sar, x_opt):
        f_sar = self.sar_backbone(x_sar)
        f_opt = self.opt_backbone(x_opt)
        f_fused, gate_weights = self.fusion_block(f_sar, f_opt)
        logits = self.classifier(f_fused)
        return logits, gate_weights

raw_model = SatQueryMultimodalFusionModel(num_classes=len(CORINE_19_CLASSES)).to(DEVICE)
model = raw_model
print("🚀 Multimodal Fusion Model successfully initialized!")


In [ ]:
# 7. ZIP Archiving & Browser Download Utilities
from IPython.display import display, HTML, Javascript

def package_and_zip_artifacts(zip_path, files_dict):
    temp_zip = zip_path + ".tmp"
    with zipfile.ZipFile(temp_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for arcname, disk_path in files_dict.items():
            if disk_path and os.path.exists(disk_path):
                zf.write(disk_path, arcname=arcname)
    if os.path.exists(zip_path):
        os.remove(zip_path)
    os.rename(temp_zip, zip_path)
    zip_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"📦 Artifacts zipped into {zip_path} ({zip_mb:.1f} MB)")
    return zip_path

def trigger_download_ui(zip_path, auto_click=True):
    if not os.path.exists(zip_path):
        return
    zip_name = os.path.basename(zip_path)
    zip_mb = os.path.getsize(zip_path) / (1024 * 1024)
    html_ui = f"""
    <div style="border: 2px solid #2563eb; border-radius: 12px; padding: 20px; margin: 15px 0; background: #0f172a; color: #f8fafc; font-family: sans-serif;">
        <div style="display: flex; align-items: center; justify-content: space-between;">
            <div>
                <h3 style="margin: 0 0 6px 0; color: #60a5fa; font-size: 1.2rem;">📥 Fusion Model Progress Saved & Zipped!</h3>
                <p style="margin: 0; color: #cbd5e1; font-size: 0.95rem;">Artifact: <b>{zip_name}</b> ({zip_mb:.1f} MB)</p>
            </div>
            <div>
                <a id="fusion-download-btn" href="{zip_name}" download="{zip_name}" 
                   style="display: inline-block; background: #2563eb; color: #ffffff; text-decoration: none; padding: 12px 22px; border-radius: 8px; font-weight: 600; font-size: 1rem;">
                    ⬇️ Download Zip Archive
                </a>
            </div>
        </div>
        <div style="margin-top: 10px; font-size: 0.82rem; color: #94a3b8;">
            💡 Artifact permanently stored in <code>/kaggle/working/{zip_name}</code> (accessible in Kaggle Output pane).
        </div>
    </div>
    """
    display(HTML(html_ui))
    if auto_click:
        display(Javascript(f"setTimeout(function() {{ var btn = document.getElementById('fusion-download-btn'); if (btn) btn.click(); }}, 1200);"))


In [ ]:
# 8. Continuous Multi-Session Training Engine
from sklearn.metrics import f1_score, average_precision_score

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(raw_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

amp_enabled = (DEVICE == "cuda")
major, _ = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
# T4 Safe: Float16 for Turing (7.5)
amp_dtype = torch.bfloat16 if (amp_enabled and major >= 8 and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=(amp_enabled and amp_dtype == torch.float16))
torch.backends.cudnn.benchmark = False

print(f"🚀 Multimodal Engine Config: Device={DEVICE}, AMP={amp_enabled}, Dtype={amp_dtype}")

# State tracking
start_shard_idx = 0
start_epoch = 1
best_val_f1 = -1.0
training_history = []

# Resume from existing staged checkpoint if available
resume_path = LATEST_CHECKPOINT_PATH if os.path.exists(LATEST_CHECKPOINT_PATH) else None
if resume_path:
    print(f"🔄 Checkpoint found at {resume_path} — restoring continuous training state...")
    try:
        ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        raw_model.load_state_dict(ckpt['model_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scaler_state_dict' in ckpt and ckpt['scaler_state_dict'] and scaler.is_enabled():
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        saved_shard = ckpt.get('shard_index', 0)
        saved_epoch = ckpt.get('epoch', 0)
        best_val_f1 = ckpt.get('best_val_macro_f1', -1.0)
        
        # If the saved epoch completed the shard, advance to the next shard
        if saved_epoch >= EPOCHS_PER_SHARD:
            start_shard_idx = saved_shard + 1
            start_epoch = 1
        else:
            start_shard_idx = saved_shard
            start_epoch = saved_epoch + 1
            
        print(f"   ✅ Auto-detected resume: Global Shard {start_shard_idx + 1}/{TOTAL_GLOBAL_SHARDS}, Epoch {start_epoch}! Best Macro-F1: {best_val_f1:.4f}")
    except Exception as e:
        print(f"   ⚠️ Checkpoint restore warning: {e} — starting fresh from Shard 1.")

# Apply user manual override if specified (e.g. FORCE_START_SHARD = None)
if FORCE_START_SHARD is not None and FORCE_START_SHARD > 0:
    start_shard_idx = FORCE_START_SHARD - 1
    start_epoch = 1
    print(f"🎯 [SHARD OVERRIDE APPLIED]: Training will commence at Global Shard {FORCE_START_SHARD} (Epoch 1) using loaded checkpoint weights!")

if os.path.exists(HISTORY_PATH):
    try:
        with open(HISTORY_PATH, 'r') as f:
            training_history = json.load(f)
        # If overriding to Shard 3, prune frozen entries from Shard >= 3 so history stays clean
        if FORCE_START_SHARD is not None and FORCE_START_SHARD > 1:
            training_history = [e for e in training_history if e.get('shard', 0) < FORCE_START_SHARD]
            print(f"   📊 Retained {len(training_history)} history records from Shards 1 to {FORCE_START_SHARD - 1}.")
    except Exception:
        training_history = []

def save_current_state(shard_idx, epoch, is_best, val_metrics=None, is_paused=False):
    val_metrics = val_metrics or {}
    state = {
        'shard_index': shard_idx,
        'epoch': epoch,
        'model_state_dict': raw_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler.is_enabled() else None,
        'best_val_macro_f1': best_val_f1,
        'val_loss': val_metrics.get('loss', None),
        'val_macro_f1': val_metrics.get('macro_f1', None),
        'val_micro_f1': val_metrics.get('micro_f1', None),
        'val_mAP': val_metrics.get('mAP', None),
        'classes': CORINE_19_CLASSES,
        'num_classes': len(CORINE_19_CLASSES),
        'is_best_so_far': is_best,
        'session_paused': is_paused,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
    }
    torch.save(state, LATEST_CHECKPOINT_PATH)
    if is_best:
        torch.save(state, BEST_CHECKPOINT_PATH)
    with open(HISTORY_PATH, 'w') as f:
        json.dump(training_history, f, indent=2)
    best_file = BEST_CHECKPOINT_PATH if os.path.exists(BEST_CHECKPOINT_PATH) else LATEST_CHECKPOINT_PATH
    package_and_zip_artifacts(ZIP_OUTPUT_PATH, {
        "fusion_resnet50_best.pt": best_file,
        "fusion_resnet50_latest.pt": LATEST_CHECKPOINT_PATH,
        "fusion_training_history.json": HISTORY_PATH,
        "classes.json": CLASSES_PATH
    })

# ----------------- Multi-Session Training Loop -----------------
training_interrupted = False
reason_for_stop = ""

# Guard: Check if all global shards in the dataset are already completed
if start_shard_idx >= TOTAL_GLOBAL_SHARDS:
    print("\n" + "="*75)
    print(f"🏆 ALL {TOTAL_GLOBAL_SHARDS} GLOBAL SHARDS ALREADY TRAINED TO COMPLETION!")
    print(f"   Checkpoint is at Global Shard {start_shard_idx}, and {TOTAL_GLOBAL_SHARDS} shards exist in attached dataset.")
    print(f"   Overall Best Macro-F1: {best_val_f1:.4f}")
    print("   No further training required. Artifacts are complete.")
    print("="*75)
    trigger_download_ui(ZIP_OUTPUT_PATH, auto_click=False)
else:
    session_end_shard_idx = min(start_shard_idx + SHARDS_PER_SESSION, TOTAL_GLOBAL_SHARDS)
    shards_to_train_count = session_end_shard_idx - start_shard_idx
    reason_for_stop = f"completed session target ({shards_to_train_count} shards)"

    print("\n" + "="*75)
    print("🎯 COMMENCING SESSION TRAINING WINDOW:")
    print(f"   • Target Range: Global Shards {start_shard_idx + 1} to {session_end_shard_idx} (Total Dataset: {TOTAL_GLOBAL_SHARDS} Shards)")
    print(f"   • Starting at: Global Shard {start_shard_idx + 1}, Epoch {start_epoch}")
    print(f"   • Shards to train this session: {shards_to_train_count}")
    print("="*75)

    try:
        for shard_idx in range(start_shard_idx, session_end_shard_idx):
            elapsed = time.time() - SESSION_START_TIME
            if elapsed >= MAX_RUNTIME_SECONDS:
                reason_for_stop = f"reached {MAX_RUNTIME_HOURS}h session safety limit"
                training_interrupted = True
                break

            shard_df = shards_meta[shard_idx]
            if len(shard_df) == 0:
                print(f"⚠️ Global Shard {shard_idx + 1} has 0 samples — skipping.")
                continue

            print(f"\n{'='*70}")
            print(f"🌐 COMMENCING GLOBAL SHARD {shard_idx + 1}/{TOTAL_GLOBAL_SHARDS} ({len(shard_df):,} paired satellite scenes)")
            print(f"{'='*70}")

            # 1. Unpack targeted 35k patches into /tmp (automatically clears previous shard!)
            extract_shard_data(shard_df, shard_idx + 1, total_shards=TOTAL_GLOBAL_SHARDS)

            # 2. Split train/val (85/15 split with guard for small remnants)
            cut = int(0.85 * len(shard_df))
            if cut == len(shard_df) and len(shard_df) > 1:
                cut = len(shard_df) - 1
            train_part = shard_df.iloc[:cut].reset_index(drop=True)
            val_part = shard_df.iloc[cut:].reset_index(drop=True)

            train_ds = BigEarthNetFusionDataset(train_part, S1_EXTRACT_DIR, S2_EXTRACT_DIR, is_train=True)
            val_ds = BigEarthNetFusionDataset(val_part, S1_EXTRACT_DIR, S2_EXTRACT_DIR, is_train=False)
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

            # CRITICAL FIX: Reset learning rate in optimizer param groups before initializing scheduler!
            # Otherwise CosineAnnealingLR captures the decayed 0.0 LR from the previous shard as base_lrs.
            for param_group in optimizer.param_groups:
                param_group['lr'] = LEARNING_RATE

            steps_per_epoch = max(1, len(train_loader))
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=EPOCHS_PER_SHARD * steps_per_epoch, eta_min=1e-6
            )
            
            # Start epoch for this shard
            current_shard_start_epoch = start_epoch if shard_idx == start_shard_idx else 1

            # If resuming mid-shard, align scheduler step
            if current_shard_start_epoch > 1:
                align_steps = (current_shard_start_epoch - 1) * steps_per_epoch
                for _ in range(align_steps):
                    scheduler.step()

            for epoch in range(current_shard_start_epoch, EPOCHS_PER_SHARD + 1):
                elapsed = time.time() - SESSION_START_TIME
                if elapsed >= MAX_RUNTIME_SECONDS:
                    reason_for_stop = f"reached {MAX_RUNTIME_HOURS}h session safety limit"
                    training_interrupted = True
                    break

                model.train()
                total_train_loss = 0.0
                t0 = time.time()
                print(f"\n--- [Global Shard {shard_idx + 1}/{TOTAL_GLOBAL_SHARDS}] Epoch [{epoch}/{EPOCHS_PER_SHARD}] (Elapsed Session: {elapsed/3600:.2f}h) ---")

                for step, (s1_imgs, s2_imgs, targets) in enumerate(train_loader):
                    s1_imgs = s1_imgs.to(DEVICE, non_blocking=True)
                    s2_imgs = s2_imgs.to(DEVICE, non_blocking=True)
                    targets = targets.to(DEVICE, non_blocking=True)
                    optimizer.zero_grad()

                    with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=amp_enabled):
                        logits, _ = model(s1_imgs, s2_imgs)
                        loss = criterion(logits, targets)

                    if scaler.is_enabled():
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        loss.backward()
                        optimizer.step()

                    scheduler.step()
                    total_train_loss += loss.item()

                    if step == 0 or (step + 1) % 50 == 0 or (step + 1) == len(train_loader):
                        lr_curr = optimizer.param_groups[0]['lr']
                        vram_mb = torch.cuda.memory_allocated(0)/1e6 if torch.cuda.is_available() else 0
                        print(f"Shard {shard_idx+1} | Ep [{epoch}/{EPOCHS_PER_SHARD}] | Step [{step+1}/{len(train_loader)}] | Loss: {loss.item():.4f} | VRAM: {vram_mb:.0f}MB | LR: {lr_curr:.2e} | Elapsed: {time.time()-t0:.1f}s", flush=True)

                # --- Validation Evaluation ---
                model.eval()
                total_val_loss = 0.0
                all_preds, all_targets = [], []
                with torch.no_grad():
                    for s1_imgs, s2_imgs, targets in val_loader:
                        s1_imgs = s1_imgs.to(DEVICE, non_blocking=True)
                        s2_imgs = s2_imgs.to(DEVICE, non_blocking=True)
                        targets = targets.to(DEVICE, non_blocking=True)
                        with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=amp_enabled):
                            logits, _ = model(s1_imgs, s2_imgs)
                            loss = criterion(logits, targets)
                        total_val_loss += loss.item()
                        all_preds.append(torch.sigmoid(logits).float().cpu().numpy())
                        all_targets.append(targets.cpu().numpy())

                # Remnant & empty val guard
                if len(val_loader) == 0 or len(all_preds) == 0:
                    avg_val_loss = 0.0
                    macro_f1, micro_f1, mAP = 0.0, 0.0, 0.0
                else:
                    avg_val_loss = total_val_loss / max(1, len(val_loader))
                    all_preds = np.concatenate(all_preds, axis=0)
                    all_targets = np.concatenate(all_targets, axis=0)
                    binary_preds = (all_preds > 0.5).astype(int)

                    macro_f1 = f1_score(all_targets, binary_preds, average='macro', zero_division=0)
                    micro_f1 = f1_score(all_targets, binary_preds, average='micro', zero_division=0)
                    try:
                        mAP = average_precision_score(all_targets, all_preds, average='macro')
                    except Exception:
                        mAP = float('nan')

                is_best = macro_f1 > best_val_f1
                if is_best:
                    best_val_f1 = macro_f1

                print(f"🏁 [Global Shard {shard_idx+1}] Epoch [{epoch}/{EPOCHS_PER_SHARD}] | Val Loss: {avg_val_loss:.4f} | Macro-F1: {macro_f1:.4f} | Micro-F1: {micro_f1:.4f} | mAP: {mAP:.4f}{' 🌟 (NEW BEST)' if is_best else ''}", flush=True)

                training_history.append({
                    'shard': shard_idx + 1,
                    'epoch': epoch,
                    'train_loss': total_train_loss / max(1, len(train_loader)),
                    'val_loss': avg_val_loss,
                    'val_macro_f1': float(macro_f1),
                    'val_micro_f1': float(micro_f1),
                    'val_mAP': float(mAP),
                    'is_best': is_best,
                    'best_macro_f1_so_far': float(best_val_f1)
                })

                save_current_state(shard_idx, epoch, is_best, {
                    'loss': avg_val_loss, 'macro_f1': macro_f1, 'micro_f1': micro_f1, 'mAP': mAP
                })

            if training_interrupted:
                break

            print(f"\n🧹 [Global Shard {shard_idx + 1} COMPLETE] Clearing /tmp to free disk space...")
            clear_fusion_tmp_dir()

    except KeyboardInterrupt:
        training_interrupted = True
        reason_for_stop = "manual Stop / KeyboardInterrupt"
        print("\n🛑 KEYBOARD INTERRUPT DETECTED (Stop button pressed)! Saving state...", flush=True)
        curr_s = shard_idx if 'shard_idx' in locals() else start_shard_idx
        curr_e = epoch if 'epoch' in locals() else start_epoch
        save_current_state(curr_s, curr_e, is_best=False, is_paused=True)

    print("\n===============================================================", flush=True)
    if training_interrupted:
        print(f"⏸️ MULTI-SHARD SESSION PAUSED ({reason_for_stop}). Artifacts safely packaged.", flush=True)
    elif session_end_shard_idx >= TOTAL_GLOBAL_SHARDS:
        print(f"🎉 ALL {TOTAL_GLOBAL_SHARDS} GLOBAL SHARDS ({len(shuffled_all):,} SCENES) TRAINED TO COMPLETION!", flush=True)
    else:
        print(f"🏁 SESSION COMPLETE: Trained up to Global Shard {session_end_shard_idx}/{TOTAL_GLOBAL_SHARDS}!", flush=True)
        print(f"👉 Next session will automatically resume at Global Shard {session_end_shard_idx + 1}.", flush=True)
    print(f"🏆 Overall Best Macro-F1: {best_val_f1:.4f}", flush=True)
    print("===============================================================", flush=True)
    trigger_download_ui(ZIP_OUTPUT_PATH, auto_click=True)
